#### 연습
- data 폴더 안에 가전 폴더의 모든 json파일을 하나의 데이터프레임으로 단순 행 결합
- 데이터에서 결측치를 확인
    - GeneralPolaruty 컬럼에서 결측치 발견
- 결측치가 포함된 데이터를 따로 저장 (na_df)
- 결측치를 제거
- 'RawText', 'GeneralPolarity' 컬럼을 제거한 나머지 컬럼 제외
- 'GeneralPolarity' 컬럼의 이름을 labels 변경
- RawText는 텍스트 정규화(특수문자 제거, 2칸 이상의 공백 제외, 문자열 앞 뒤 공백 제거)
- labels 데이터에서 -1 과 0 은 0으로, 1은 1로 데이터를 변경 -> 해당 컬럼의 dtype을
int 변경 -> BERTmodel에서 선형 모델로 확률을 예측하기 때문에 labels가 위치값
- train, test 형태로 데이터를 9:1 비율로 나눠준다.
    - labels를 기준으로 계층화 분할
- 데이터프레임을 Dataset의 형태로 변환
- token화 작업은 AutoTokenizer를 이용하여 모델의 이름은 skt/kobert-base-v1 이용하여 토큰화
- 같은 모델을 로드하여 BERTModel + Linear 모델 정의
- Trainer, TrainingArguments를 이용하여 학습
- 학습 -> na_df에서 상위 5개의 RawText을 이용하여 예측

In [6]:
import pandas as pd
import re
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

In [7]:
# 정규화 함수
def normalize_token_text(text : str) -> str:
    text = re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [9]:
import glob # 여러개의 데이터로드

In [10]:
# 데이터 로드
path_to_json = '../data/가전/*.json'
file_list = glob.glob(path_to_json)

In [ ]:
df_list = [pd.read_json(file) for file in file_list]
df_list

In [16]:
df = pd.concat(df_list, ignore_index=True)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4056 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 380.4+ KB


In [24]:
missing_df = df[df.isna().any(axis=1)].copy()

In [ ]:
missing_df

In [27]:
df = df.dropna()
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            3678 non-null   int64  
 1   RawText          3678 non-null   object 
 2   Source           3678 non-null   object 
 3   Domain           3678 non-null   object 
 4   MainCategory     3678 non-null   object 
 5   ProductName      3678 non-null   object 
 6   ReviewScore      3678 non-null   int64  
 7   Syllable         3678 non-null   int64  
 8   Word             3678 non-null   int64  
 9   RDate            3678 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          3678 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 373.5+ KB


In [38]:
df = df[['RawText', 'GeneralPolarity']]
df.rename(columns={
    'GeneralPolarity' : 'labels'
}, inplace=True)

In [ ]:
df['RawText'] = df['RawText'].map(normalize_token_text) # 정규화

In [ ]:
df['labels'].value_counts()

labels
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [49]:
df['labels'].replace(-1, 0, inplace=True)

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 4055
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   labels   3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 86.2+ KB


In [58]:
df['labels'] = df['labels'].astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 4055
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   RawText  3678 non-null   object
 1   labels   3678 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 86.2+ KB


In [59]:
from datasets import Dataset

In [60]:
train_df, test_df = train_test_split(
    df, test_size=0.1, random_state=42, stratify=df['labels']
)
# BERT 모델에서 사용하는 데이터 타입으로 변경
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

In [61]:
train_ds

Dataset({
    features: ['RawText', 'labels'],
    num_rows: 3310
})

In [62]:
MODEL_NAME = "skt/kobert-base-v1"

# use_fast = False -> 기본(파이썬 기반) 토크나이저
    # KoBERT 모델은 sentencepiece 기반 토큰화
# use_fast = True -> 빠른 토크나이저 -> Rust, C 기반
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = False)

In [63]:
# 토크나이저 함수 (RawText 컬럼 사용)
def tok_fn(batch):
    return tokenizer(batch['RawText'], truncation=True, max_length=128, padding='max_length')

In [64]:
# 토큰화 및 불필요 컬럼 제거
train_tok = train_ds.map(tok_fn, batched=True, remove_columns=['RawText'])
test_tok = test_ds.map(tok_fn, batched=True, remove_columns=['RawText'])

Map: 100%|██████████| 368/368 [00:00<00:00, 2639.41 examples/s]


In [65]:
# 모델 정의 (01_BERT의 BERTClsHead 참고)
class BERTClsHead(nn.Module):
    def __init__(self, model_name, num_label=2, dropout=0.1):
        super().__init__()
        self.backbone = BertModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_label)
        # tokenizer는 노트북에서 미리 로드되어 있음
        self.backbone.config.pad_token_id = tokenizer.pad_token_id

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]
        drop_out_data = self.dropout(pooled)
        logits = self.classifier(drop_out_data)
        result = {'logits': logits}
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            result['loss'] = loss
        return result

In [ ]:
# 모델 생성
model = BERTClsHead(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
model.to(device)

In [67]:
# 추가 셀: 필요한 임포트
from sklearn.metrics import accuracy_score, f1_score

In [68]:
# 평가 지표
def metrics(eval_pred):
    logits, y = eval_pred
    preds = logits.argmax(-1)
    return {
        'accuracy': accuracy_score(y, preds),
        'f1': f1_score(y, preds, average='binary' if len(set(y))==2 else 'macro')
    }

In [69]:
# 학습 설정 (필요에 맞게 조정)
args = TrainingArguments(
    output_dir="./kobert_exam_out",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to=[]
)


In [70]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    compute_metrics=metrics
)

C:\Users\johnh\AppData\Local\Temp\ipykernel_32648\2884272174.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [72]:
# 평가 (학습 후 또는 사전 학습 모델 평가)
eval_res = trainer.evaluate()
print("평가 결과:", eval_res)

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


평가 결과: {'eval_loss': 0.6950284838676453, 'eval_model_preparation_time': 0.0021, 'eval_accuracy': 0.5244565217391305, 'eval_f1': 0.6391752577319587, 'eval_runtime': 21.5481, 'eval_samples_per_second': 17.078, 'eval_steps_per_second': 1.067}


In [73]:
samples = missing_df['RawText'].head(5).map(normalize_token_text).tolist()
enc = tokenizer(samples, return_tensors='pt', padding=True, truncation=True, max_length=128)

In [75]:
with torch.no_grad():
        out = model(**enc)
        probs = torch.softmax(out['logits'], dim=-1).cpu().numpy()

In [78]:
for s, p in zip(samples, probs):
    print(f"{s[:15]} -> Negative:{p[0]:.3f}, Positive:{p[1]:.3f}, Pred:{p.argmax()}")

귀에서 자꾸 빠져요.귀에 꼽 -> Negative:0.489, Positive:0.511, Pred:1
아이를 출산한 기념으로 TV -> Negative:0.490, Positive:0.510, Pred:1
화면에 노이즈가 생깁니다.  -> Negative:0.519, Positive:0.481, Pred:0
이번에 이사하면서 우리 따님 -> Negative:0.495, Positive:0.505, Pred:1
기존에 사용하던 무선이어폰이 -> Negative:0.460, Positive:0.540, Pred:1
